# Neural Network xG Model

In [193]:
import pandas as pd
from football_analytics.utils import parsing, supabase, shot_geometry
import football_analytics.visuals.shots as shots
from sklearn.metrics import log_loss
from importlib import reload
import numpy as np
from sklearn.model_selection import train_test_split
import numpy as np
import json
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import joblib
reload(parsing)
reload(supabase)
reload(shot_geometry)
reload(shots)

<module 'football_analytics.visuals.shots' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\visuals\\shots.py'>

### Load Supabase Data

In [194]:
table_name = 'shots'
key_column = 'statsbomb_event_id'

In [195]:
data = supabase.fetch_all_rows_in_batches(table_name=table_name, key_column=key_column, batch_size=2000)

Fetched 2000 rows (total 2000).
Fetched 2000 rows (total 4000).
Fetched 2000 rows (total 6000).
Fetched 2000 rows (total 8000).
Fetched 2000 rows (total 10000).
Fetched 2000 rows (total 12000).
Fetched 2000 rows (total 14000).
Fetched 2000 rows (total 16000).
Fetched 2000 rows (total 18000).
Fetched 2000 rows (total 20000).
Fetched 2000 rows (total 22000).
Fetched 2000 rows (total 24000).
Fetched 2000 rows (total 26000).
Fetched 2000 rows (total 28000).
Fetched 2000 rows (total 30000).
Fetched 2000 rows (total 32000).
Fetched 2000 rows (total 34000).
Fetched 2000 rows (total 36000).
Fetched 2000 rows (total 38000).
Fetched 2000 rows (total 40000).
Fetched 2000 rows (total 42000).
Fetched 2000 rows (total 44000).
Fetched 2000 rows (total 46000).
Fetched 2000 rows (total 48000).
Fetched 2000 rows (total 50000).
Fetched 2000 rows (total 52000).
Fetched 2000 rows (total 54000).
Fetched 2000 rows (total 56000).
Fetched 2000 rows (total 58000).
Fetched 2000 rows (total 60000).
Fetched 2000 r

In [196]:
df_raw = pd.DataFrame(data)

In [197]:
data_players = supabase.fetch_all_rows_in_batches(
    table_name="players",
    key_column="statsbomb_player_id",
    batch_size=2000,
    columns="statsbomb_player_id, position_id, position_name",
)

Fetched 2000 rows (total 2000).
Fetched 2000 rows (total 4000).
Fetched 2000 rows (total 6000).
Fetched 2000 rows (total 8000).
Fetched 1043 rows (total 9043).


In [198]:
df_players = pd.DataFrame(data_players)

In [199]:
# Merging player data with shots data on player ID
df_merged = df_raw.merge(df_players, left_on='shot_taker_id', right_on='statsbomb_player_id', how='left')

In [200]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88023 entries, 0 to 88022
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  88023 non-null  object 
 1   statsbomb_event_id          88023 non-null  object 
 2   x1                          88023 non-null  float64
 3   y1                          88023 non-null  float64
 4   distance_to_goal            88023 non-null  float64
 5   angle_to_goal_deg           88023 non-null  float64
 6   keeper_distance             88023 non-null  float64
 7   min_defender_distance       88015 non-null  float64
 8   avg_defender_distance       88015 non-null  float64
 9   num_def_in_shot_triangle    88023 non-null  int64  
 10  num_teammates_in_box        88023 non-null  int64  
 11  shot_to_min_def_ratio       88015 non-null  float64
 12  shot_type                   88023 non-null  object 
 13  body_part                   880

In [201]:
data_teams = supabase.fetch_all_rows_in_batches(
    table_name="teams",
    key_column="team_id",
    batch_size=2000,
    columns="team_id, team_gender, team_name",
)
df_teams = pd.DataFrame(data_teams)
df_merged = df_merged.merge(df_teams, left_on='attacking_team_id', right_on='team_id', how='left')

Fetched 312 rows (total 312).


In [202]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88023 entries, 0 to 88022
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  88023 non-null  object 
 1   statsbomb_event_id          88023 non-null  object 
 2   x1                          88023 non-null  float64
 3   y1                          88023 non-null  float64
 4   distance_to_goal            88023 non-null  float64
 5   angle_to_goal_deg           88023 non-null  float64
 6   keeper_distance             88023 non-null  float64
 7   min_defender_distance       88015 non-null  float64
 8   avg_defender_distance       88015 non-null  float64
 9   num_def_in_shot_triangle    88023 non-null  int64  
 10  num_teammates_in_box        88023 non-null  int64  
 11  shot_to_min_def_ratio       88015 non-null  float64
 12  shot_type                   88023 non-null  object 
 13  body_part                   880

## Prepare Data

In [203]:
df = df_merged.dropna().copy()

In [204]:
df['goal_outcome'] = df['outcome'].apply(lambda x: 1 if x=='Goal' else 0)
df['is_with_feet'] = df['body_part'].apply(lambda x: 1 if (x=='Right Foot' or x=='Left Foot') else 0)
df['is_penalty'] = df['shot_type'].apply(lambda x: 1 if x=='Penalty' else 0)

In [205]:
df['is_defender'] = df['position_id'].apply(lambda x: 1 if x in [1,2,3,4,5,6,7,8] else 0)
df['is_midfielder'] = df['position_id'].apply(lambda x: 1 if x in [9,10,11,12,13,14,15,16,17,18,19,20] else 0)
df['is_forward'] = df['position_id'].apply(lambda x: 1 if x in [21,22,23,24,25] else 0)

In [206]:
df['is_male'] = df['team_gender'].apply(lambda x: 1 if x=='male' else 0)

In [207]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87075 entries, 0 to 88022
Data columns (total 43 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   created_at                  87075 non-null  object 
 1   statsbomb_event_id          87075 non-null  object 
 2   x1                          87075 non-null  float64
 3   y1                          87075 non-null  float64
 4   distance_to_goal            87075 non-null  float64
 5   angle_to_goal_deg           87075 non-null  float64
 6   keeper_distance             87075 non-null  float64
 7   min_defender_distance       87075 non-null  float64
 8   avg_defender_distance       87075 non-null  float64
 9   num_def_in_shot_triangle    87075 non-null  int64  
 10  num_teammates_in_box        87075 non-null  int64  
 11  shot_to_min_def_ratio       87075 non-null  float64
 12  shot_type                   87075 non-null  object 
 13  body_part                   87075 no

In [208]:
feature_columns = [
    'distance_to_goal',
    'angle_to_goal_deg',
    'keeper_distance',
    'min_defender_distance',
    'num_teammates_in_box',
    'is_penalty',
    'num_def_in_shot_triangle',
    'frac_goal_uncovered',
    'keeper_is_in_shot_triangle',
    'is_with_feet',
    'under_pressure',
    'is_defender',
    'is_midfielder',
    'is_forward',
    'is_male'
    ]

In [209]:
X = df[feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
y = np.array(df['goal_outcome'])

#### Check for NaN and Inf in Input / Output

In [210]:
if np.isnan(y).sum() > 0:
    raise ValueError("NaN values found in target variable y")

In [211]:
if np.isnan(X).sum() > 0:
    raise ValueError("NaN values found in feature matrix X")

## Define Loader, Model and Optimizer

In [212]:
scaler = StandardScaler().fit(X)
X = scaler.transform(X)

In [213]:
class ShotDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [214]:
dataset = ShotDataset(X, y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [215]:
import torch.nn as nn

class XGModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)  # output = logit
        )

    def forward(self, x):
        return self.net(x)


In [224]:
model = XGModel(input_dim=X.shape[1])

In [225]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [226]:
np.isnan(X).sum(), np.isinf(X).sum()

(np.int64(0), np.int64(0))

In [227]:
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss / len(loader):.4f}")


Epoch 01 | Loss: 0.2942
Epoch 02 | Loss: 0.2705
Epoch 03 | Loss: 0.2691
Epoch 04 | Loss: 0.2684
Epoch 05 | Loss: 0.2679
Epoch 06 | Loss: 0.2673
Epoch 07 | Loss: 0.2670
Epoch 08 | Loss: 0.2665
Epoch 09 | Loss: 0.2665
Epoch 10 | Loss: 0.2660
Epoch 11 | Loss: 0.2660
Epoch 12 | Loss: 0.2658
Epoch 13 | Loss: 0.2656
Epoch 14 | Loss: 0.2655
Epoch 15 | Loss: 0.2653
Epoch 16 | Loss: 0.2648
Epoch 17 | Loss: 0.2649
Epoch 18 | Loss: 0.2646
Epoch 19 | Loss: 0.2644
Epoch 20 | Loss: 0.2642
Epoch 21 | Loss: 0.2639
Epoch 22 | Loss: 0.2638
Epoch 23 | Loss: 0.2638
Epoch 24 | Loss: 0.2636
Epoch 25 | Loss: 0.2633
Epoch 26 | Loss: 0.2632
Epoch 27 | Loss: 0.2630
Epoch 28 | Loss: 0.2627
Epoch 29 | Loss: 0.2626
Epoch 30 | Loss: 0.2623
Epoch 31 | Loss: 0.2623
Epoch 32 | Loss: 0.2618
Epoch 33 | Loss: 0.2621
Epoch 34 | Loss: 0.2617
Epoch 35 | Loss: 0.2615
Epoch 36 | Loss: 0.2615
Epoch 37 | Loss: 0.2613
Epoch 38 | Loss: 0.2612
Epoch 39 | Loss: 0.2606
Epoch 40 | Loss: 0.2608
Epoch 41 | Loss: 0.2606
Epoch 42 | Loss:

In [228]:
def xg_log_loss(y_true, y_pred):
    """
    Computes binary cross-entropy (log loss) for xG predictions.
    """
    return log_loss(y_true, y_pred)

In [229]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X, dtype=torch.float32))
    xg_probs = torch.sigmoid(logits).numpy().flatten()


In [230]:
xg_log_loss_nn = xg_log_loss(y, xg_probs)
print(f"NN xG log loss: {xg_log_loss_nn:.4f}")

NN xG log loss: 0.2576


## Save Model

In [231]:
from pathlib import Path
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f"nn_xg_model_{date_str}.pth"
torch.save(model.state_dict(), f'../nn_models/{model_name}')
joblib.dump(scaler, f'../nn_models/scaler_{date_str}.save')

def save_feature_columns(columns, output_path):
    payload = {"feature_columns": list(columns)}
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


feature_columns_path = f"../nn_models/config_{date_str}.json"
save_feature_columns(feature_columns, feature_columns_path)

In [232]:
df['nn_xg'] = xg_probs

In [233]:
df[['nn_xg', 'statsbomb_xg','statsbomb_event_id']].to_csv('nn_xg_model_predictions.csv', index=False)

## Check Differences

In [242]:
df.head()

,created_at,statsbomb_event_id,x1,y1,distance_to_goal,angle_to_goal_deg,keeper_distance,min_defender_distance,avg_defender_distance,num_def_in_shot_triangle,...,team_gender,team_name,goal_outcome,is_with_feet,is_penalty,is_defender,is_midfielder,is_forward,is_male,nn_xg
0,2025-12-08T21:04:18.607957+00:00,00006e24-97ca-455a-b9e1-ce12770a34c7,98.7,59.4,28.810588,11.822664,22.663186,9.656604,12.041987,0,...,female,Norway Women's,0,1,0,0,1,0,0,0.077693
1,2025-12-08T21:09:45.845141+00:00,00007fd2-738e-4b27-bb68-7bdeffc77ee7,101.2,24.0,24.686839,14.221466,22.145203,0.921954,12.861515,0,...,male,Espanyol,0,1,0,0,0,1,1,0.025869
2,2025-12-08T21:09:47.691664+00:00,0000ed2d-ee36-4038-bfec-1083ecc52a8d,92.0,31.0,29.410882,14.780097,28.792360,7.071068,16.531788,2,...,female,OL Reign,0,1,0,0,1,0,0,0.011529
3,2025-12-08T20:53:26.055264+00:00,00015bc8-c90f-43dc-a99a-304be6016fcd,93.6,60.0,33.120386,11.055286,30.876852,2.915476,9.428918,1,...,male,East Bengal,0,1,0,0,1,0,1,0.007812
4,2025-12-08T21:07:13.279534+00:00,00018844-2ac3-441c-b751-8ed47ad26daf,100.4,44.1,20.024235,22.161287,17.525125,1.131371,8.147923,2,...,male,Belgium,1,1,0,0,1,0,1,0.029968


In [243]:
event_id = "00097113-ec14-437e-8ca4-de132757678e"
shot = df[df['statsbomb_event_id'] == event_id]

In [246]:
shot[feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

array([[ 9.98248466, 32.65033086,  7.28010989,  3.61247837,  1.        ,
         0.        ,  0.        ,  0.63779071,  1.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ]])

In [247]:
X_temp = (
    shot[feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
)

In [249]:
X_temp

array([[ 9.98248466, 32.65033086,  7.28010989,  3.61247837,  1.        ,
         0.        ,  0.        ,  0.63779071,  1.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  1.        ]])

In [250]:
X_temp = scaler.transform(X_temp)

In [251]:
X_temp

array([[-1.05190527,  0.45774253, -1.00276517,  0.2004497 , -0.02160389,
        -0.12539671, -0.77832149,  0.42428961,  0.21000549,  0.43885742,
        -0.55703245,  2.19291418, -0.89644663, -0.78638679,  0.44038249]])

In [252]:
with torch.no_grad():
    logits = model(torch.tensor(X_temp, dtype=torch.float32))
    xg_temp = torch.sigmoid(logits).numpy().flatten()

In [253]:
xg_temp

array([0.3397009], dtype=float32)

In [254]:
shot_dict = parsing.parse_json_field(shot.iloc[0]['full_json'])

In [255]:
fig = shots.plot_shot_details(shot_dict, show=True)